# Continuum — Pi Script + Rift Playground

**Pi Script** declares what must stay true about an AI system's behavior.  
**Rift** maps natural language intent to Pi Script constraints automatically.  
Together they close the loop from human declaration to machine accountability.

---

This notebook walks through the full Continuum stack interactively:

1. Write a Pi Script governance policy
2. Validate it and inspect the IR
3. Run the resolver against a state snapshot — see the RESOLUTION TRACE
4. Trigger a violation — see the system freeze
5. Write a Rift program — compile it to Pi Script automatically
6. Run the Rift-generated policy through the resolver

## Setup

Run this cell once to add the project root to the Python path.

In [1]:
import sys, json
from pathlib import Path

# Add project root so imports work
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pi_script.parser import parse_string
from pi_script.validator import PiValidator
from pi_script.resolver import resolve

print(f"Project root: {PROJECT_ROOT}")
print("Imports OK")

Project root: /home/jovyan/continuum
Imports OK


---

## Part 1 — Write a Pi Script policy

This is a governance policy for an AI task agent.  
It declares three constraints that must remain true at all times.

In [2]:
POLICY = """
domain ai_governance {
    audit_interval: 24 hours
    tiebreaker:     timestamp_asc
}

entity TaskAgent {
    confidence_score: range(0.0 .. 1.0)
    current_mode:     text
    is_active:        boolean
}

map SafeMode {
    target:   TaskAgent.current_mode
    maps_to:  "safe_mode"
    triggers: ["safe", "restricted"]
}

map NormalMode {
    target:   TaskAgent.current_mode
    maps_to:  "normal_mode"
    triggers: ["normal", "standard"]
}

constraint ConfidenceFloor {
    priority:     critical
    rule:         TaskAgent.confidence_score must remain within range(0.2 .. 1.0)
    on_violation: freeze + escalate
}

constraint ModeCompliance {
    priority:     high
    rule:         TaskAgent.current_mode must match mapped_values
    on_violation: escalate
}

constraint SessionActive {
    priority:     high
    rule:         TaskAgent.is_active must equal true
    on_violation: flag
}

enforce {
    entity:      TaskAgent
    constraints: [ConfidenceFloor, ModeCompliance, SessionActive]
}
"""

tree, err = parse_string(POLICY)
assert err is None, f"Parse error: {err}"
print("Parse OK")

Parse OK


## Part 2 — Validate and inspect the IR

The validator runs semantic checks and produces a structured IR (Intermediate Representation) — the machine-readable form of your policy.

In [3]:
ok, errors, ir = PiValidator(tree).validate()

if errors:
    for e in errors:
        print(f"ERROR  {e}")
else:
    print("Validation OK\n")
    print(f"Domain:    {ir['domain']}")
    print(f"Entity:    {list(ir['entities'].keys())}")
    print(f"Constraints: {list(ir['constraints'].keys())}")
    print(f"\nFull IR:")
    print(json.dumps(ir, indent=2))

Validation OK

Domain:    ai_governance
Entity:    ['TaskAgent']
Constraints: ['ConfidenceFloor', 'ModeCompliance', 'SessionActive']

Full IR:
{
  "domain": "ai_governance",
  "audit_interval": {
    "value": 24.0,
    "unit": "hours"
  },
  "tiebreaker": "timestamp_asc",
  "entities": {
    "TaskAgent": {
      "confidence_score": "range(0.0..1.0)",
      "current_mode": "text",
      "is_active": "boolean"
    }
  },
  "constraints": {
    "ConfidenceFloor": {
      "priority": "critical",
      "rule": {
        "kind": "range_rule",
        "ref": "TaskAgent.confidence_score",
        "lo": 0.2,
        "hi": 1.0
      },
      "on_violation": [
        "freeze",
        "escalate"
      ],
      "escalation": [],
      "decay_check": null
    },
    "ModeCompliance": {
      "priority": "high",
      "rule": {
        "kind": "membership_rule",
        "ref": "TaskAgent.current_mode"
      },
      "on_violation": [
        "escalate"
      ],
      "escalation": [],
      "decay_

## Part 3 — Run the resolver (clean state)

The resolver evaluates constraints against a state snapshot and emits a RESOLUTION TRACE.

In [4]:
clean_state = {
    "trigger_type": "event",
    "entity": "TaskAgent",
    "entity_state": {
        "confidence_score": 0.85,
        "current_mode":     "normal_mode",
        "is_active":        True,
    }
}

trace, rendered, exit_code = resolve(ir, clean_state)
print(rendered)


RESOLUTION TRACE
════════════════════════════════════════════════════════
Timestamp    : 2026-05-28T18:30:45.244Z
Domain       : ai_governance
Entity       : TaskAgent
Trigger      : event — state snapshot received for TaskAgent
════════════════════════════════════════════════════════
├── CONSTRAINT: ConfidenceFloor [priority: critical]
│   ├── Rule kind  : range_rule
│   ├── Evaluation : confidence_score = 0.85, within range(0.2 .. 1.0)
│   └── ✓ SATISFIED — no action
│
├── CONSTRAINT: ModeCompliance [priority: high]
│   ├── Rule kind  : membership_rule
│   ├── Evaluation : current_mode = 'normal_mode', matched in valid set ['normal_mode', 'safe_mode']
│   └── ✓ SATISFIED — no action
│
└── CONSTRAINT: SessionActive [priority: high]
│   ├── Rule kind  : equality_rule
│   ├── Evaluation : is_active = True, equals expected True
│   └── ✓ SATISFIED — no action
│
└── RESOLUTION
    ├── System state : running
    └── All rules were checked and passed: ConfidenceFloor, ModeCompliance, and S

## Part 4 — Trigger a violation

Drop the confidence score below the floor. Watch `ConfidenceFloor` fire at critical priority.

In [5]:
violation_state = {
    "trigger_type": "event",
    "entity": "TaskAgent",
    "entity_state": {
        "confidence_score": 0.10,   # below the 0.2 floor
        "current_mode":     "normal_mode",
        "is_active":        True,
    }
}

trace, rendered, exit_code = resolve(ir, violation_state)
print(rendered)
print(f"\nExit code: {exit_code}  (0 = clean, 1 = violation)")


RESOLUTION TRACE
════════════════════════════════════════════════════════
Timestamp    : 2026-05-28T18:30:48.027Z
Domain       : ai_governance
Entity       : TaskAgent
Trigger      : event — state snapshot received for TaskAgent
════════════════════════════════════════════════════════
├── CONSTRAINT: ConfidenceFloor [priority: critical]
│   ├── Rule kind  : range_rule
│   ├── Evaluation : confidence_score 0.1 is below minimum 0.2 (range floor)
│   ├── ✗ VIOLATION DETECTED
│   └── Action     : freeze + escalate
│
├── CONSTRAINT: ModeCompliance [priority: high]
│   ├── Rule kind  : membership_rule
│   ├── Evaluation : current_mode = 'normal_mode', matched in valid set ['normal_mode', 'safe_mode']
│   └── ✓ SATISFIED — no action
│
└── CONSTRAINT: SessionActive [priority: high]
│   ├── Rule kind  : equality_rule
│   ├── Evaluation : is_active = True, equals expected True
│   └── ✓ SATISFIED — no action
│
└── RESOLUTION
    ├── Action       : freeze + escalate
    ├── System state : frozen

---

## Part 5 — Write a Rift program

Rift lets you declare intent in plain language. This compiles automatically to Pi Script constraints — no hand-written policy needed.

In [6]:
from rift.parser import parse_string as rift_parse
from rift.validator import RiftValidator
from rift.compiler import RiftCompiler

RIFT_PROGRAM = """
map "I shelved [project]"     -> project.state: dormant
map "let's revisit [project]" -> project.state: active
map "I'm done with [project]" -> project.state: closed

intent RespectShelfedProjects {
    when user declares: "I shelved [project]"
    treat: [project] as dormant
    until: user declares "let's revisit [project]"
    enforce: "do not reference [project] unprompted"
    generates: Pi Script constraint ShelvedProjectGuard
}

intent ReactivateProject {
    when user declares: "let's revisit [project]"
    treat: [project] as active
    releases: RespectShelfedProjects for [project]
    generates: Pi Script constraint ActiveProjectGuard
}
"""

tree, err = rift_parse(RIFT_PROGRAM)
assert err is None, f"Parse error: {err}"

ok, errors, rift_ir = RiftValidator(tree).validate()
assert ok, "Validation errors: " + "\n".join(errors)

print("Rift parse + validate OK")
print(f"\nIntents:  {list(rift_ir['intents'].keys())}")
print(f"Maps:     {len(rift_ir['maps'])} declared")
print(f"\nRift IR:")
print(json.dumps(rift_ir, indent=2))

Rift parse + validate OK

Intents:  ['RespectShelfedProjects', 'ReactivateProject']
Maps:     3 declared

Rift IR:
{
  "intents": {
    "RespectShelfedProjects": {
      "when": "I shelved [project]",
      "treat": {
        "capture": "project",
        "as": "dormant"
      },
      "until": {
        "kind": "user_declares",
        "value": "let's revisit [project]"
      },
      "enforce": "do not reference [project] unprompted",
      "releases": null,
      "generates": "ShelvedProjectGuard",
      "priority": null
    },
    "ReactivateProject": {
      "when": "let's revisit [project]",
      "treat": {
        "capture": "project",
        "as": "active"
      },
      "until": null,
      "enforce": null,
      "releases": {
        "intent": "RespectShelfedProjects",
        "capture": "project"
      },
      "generates": "ActiveProjectGuard",
      "priority": null
    }
  },
  "maps": [
    {
      "pattern": "I shelved [project]",
      "target_entity": "project",
   

## Part 6 — Compile Rift → Pi Script and run the resolver

The compiler emits a valid Pi Script policy from the Rift IR. Then we run it through the resolver.

In [7]:
# Compile Rift → Pi Script source
pi_source = RiftCompiler(rift_ir, source_name="playground").compile()
print("=== Generated Pi Script ===\n")
print(pi_source)

=== Generated Pi Script ===

// Generated by Rift v0.1 compiler
// Source: playground
// Do not edit — regenerate from source.

domain rift_generated {
    audit_interval: 1 hour
    tiebreaker:     timestamp_asc
}

entity Project {
    state: text
}

map DormantMap {
    target:   Project.state
    maps_to:  "dormant"
    triggers: ["I shelved [project]"]
}

map ActiveMap {
    target:   Project.state
    maps_to:  "active"
    triggers: ["let's revisit [project]"]
}

map ClosedMap {
    target:   Project.state
    maps_to:  "closed"
    triggers: ["I'm done with [project]"]
}

constraint ShelvedProjectGuard {
    priority:     medium
    rule:         Project.state must equal "dormant"
    on_violation: escalate
}

constraint ActiveProjectGuard {
    priority:     medium
    rule:         Project.state must equal "active"
    on_violation: escalate
}

enforce {
    entity:      Project
    constraints: [ShelvedProjectGuard, ActiveProjectGuard]
}



In [8]:
# Validate the generated Pi Script and run the resolver
from pi_script.validator import validate_file as pi_validate_file

# Write to a temp file, validate, then resolve
import tempfile, os
with tempfile.NamedTemporaryFile(mode='w', suffix='.pi', delete=False, encoding='utf-8') as f:
    f.write(pi_source)
    tmp_path = f.name

ok, errors, generated_ir = pi_validate_file(tmp_path)
os.unlink(tmp_path)

assert ok, "Generated Pi Script failed validation:\n" + "\n".join(errors)
print("Generated Pi Script validates OK\n")

# Run the resolver — project state: dormant (shelved)
shelved_state = {
    "trigger_type": "event",
    "entity": "Project",
    "entity_state": {
        "state":      "dormant",
        "index_name": "veritas",
    }
}

trace, rendered, exit_code = resolve(generated_ir, shelved_state)
print(rendered)

Generated Pi Script validates OK


RESOLUTION TRACE
════════════════════════════════════════════════════════
Timestamp    : 2026-05-28T18:30:54.219Z
Domain       : rift_generated
Entity       : Project
Trigger      : event — state snapshot received for Project
════════════════════════════════════════════════════════
├── CONSTRAINT: ShelvedProjectGuard [priority: medium]
│   ├── Rule kind  : equality_rule
│   ├── Evaluation : state = 'dormant', equals expected 'dormant'
│   └── ✓ SATISFIED — no action
│
└── CONSTRAINT: ActiveProjectGuard [priority: medium]
│   ├── Rule kind  : equality_rule
│   ├── Evaluation : state is 'dormant', expected 'active'
│   ├── ✗ VIOLATION DETECTED
│   └── Action     : escalate
│
└── RESOLUTION
    ├── Action       : escalate
    ├── System state : escalated
    └── The rule 'ActiveProjectGuard' was broken: state is 'dormant', expected 'active'. This has been sent to a human reviewer. The issue has been passed to a human reviewer.

